In [4]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# ── 데이터 ──────────────────────────────────────────────
full_data = [
    ('BLCA',  105, 49,  1.03,  0.92, 1.15, 0.635),
    ('COAD',  21,  8,   1.03,  0.50, 2.10, 0.944),
    ('KIRC',  171, 47,  1.07,  0.92, 1.24, 0.385),
    ('KIRP',  22,  6,   0.95,  0.46, 1.94, 0.881),
    ('LIHC',  72,  34,  1.38,  1.18, 1.61, 0.000),
    ('STAD',  43,  28,  0.82,  0.62, 1.09, 0.173),
]
full_overall = ('Overall', 494, 192, 1.065, 1.002, 1.132, 0.042)

nokirc_data = [
    ('BLCA',  105, 49,  1.03,  0.92, 1.15, 0.635),
    ('COAD',  21,  8,   1.03,  0.50, 2.10, 0.944),
    ('KIRP',  22,  6,   0.95,  0.46, 1.94, 0.881),
    ('LIHC',  72,  34,  1.38,  1.18, 1.61, 0.000),
    ('STAD',  43,  28,  0.82,  0.62, 1.09, 0.173),
]
nokirc_overall = ('Overall', 320, 129, 1.068, 0.997, 1.143, 0.059)

XMIN, XMAX = 0.3, 2.5
X_REF = 1.0

def draw_panel(fig, gs_col, rows, overall, title):
    """
    각 패널을 3개 서브축으로 구성:
      ax_left  : Cancer Type | N (events)  텍스트
      ax_mid   : Forest plot (CI 선 + 포인트)
      ax_right : HR (95% CI), p-value 텍스트
    """
    n_rows = len(rows) + 2   # rows + 구분줄 gap + overall

    # 3-column grid: 왼쪽 텍스트 | 플롯 | 오른쪽 텍스트
    inner = gridspec.GridSpecFromSubplotSpec(
        1, 3, subplot_spec=gs_col,
        width_ratios=[1.8, 3.0, 2.2], wspace=0
    )
    ax_l = fig.add_subplot(inner[0])
    ax_m = fig.add_subplot(inner[1])
    ax_r = fig.add_subplot(inner[2])

    for ax in [ax_l, ax_m, ax_r]:
        ax.set_ylim(-0.8, n_rows + 0.5)
        ax.set_yticks([])
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_visible(False)
        ax.spines['bottom'].set_visible(False)

    # 중앙 플롯 축만 x축 표시
    ax_m.set_xlim(XMIN, XMAX)
    ax_m.spines['bottom'].set_visible(True)
    ax_m.set_xlabel('Hazard Ratio (PC6)', fontsize=9, color='#333333')
    ax_m.set_xticks([0.5, 1.0, 1.5, 2.0])
    ax_m.tick_params(axis='x', labelsize=8.5)
    ax_m.axvline(X_REF, color='#777777', lw=1.0, linestyle='--', alpha=0.7, zorder=1)

    ax_l.set_xlim(0, 1)
    ax_r.set_xlim(0, 1)
    for ax in [ax_l, ax_r]:
        ax.tick_params(left=False, bottom=False, labelbottom=False)

    # 헤더
    ax_l.text(0.04, n_rows + 0.1, 'Cancer Type',
              fontsize=8.5, fontweight='bold', color='#333333', va='bottom')
    ax_l.text(0.72, n_rows + 0.1, 'N (events)',
              fontsize=8.5, fontweight='bold', color='#333333', va='bottom', ha='center')
    ax_r.text(0.5, n_rows + 0.1, 'HR (95% CI)      p-value',
              fontsize=8.5, fontweight='bold', color='#333333', va='bottom', ha='center')

    # 제목
    ax_m.set_title(title, fontsize=9.5, fontweight='bold',
                   color='#1A3A6B', pad=10)

    def plot_row(i, label, N, events, HR, low, high, p, is_overall=False):
        y = n_rows - 1 - i  # 위→아래

        # 배경
        shade = '#F0F4FA' if (i % 2 == 0 and not is_overall) else 'white'
        if not is_overall:
            for ax in [ax_l, ax_m, ax_r]:
                ax.axhspan(y - 0.42, y + 0.42, color=shade, zorder=0)

        color  = '#1A3A6B' if is_overall else '#2874A6'
        marker = 'D'        if is_overall else 's'
        ms     = 80         if is_overall else 40
        lw     = 2.2        if is_overall else 1.5
        fs     = 9.5        if is_overall else 9.0
        fw     = 'bold'     if is_overall else 'normal'

        # 왼쪽: 암종명
        ax_l.text(0.04, y, label,
                  va='center', ha='left', fontsize=fs, fontweight=fw,
                  color='#1A3A6B' if is_overall else '#222222')

        # 왼쪽: N (events) — 오른쪽 정렬
        ax_l.text(0.96, y, f'{N} ({events})',
                  va='center', ha='right', fontsize=8.2,
                  color='#555555', style='italic' if not is_overall else 'normal',
                  fontweight=fw)

        # 중앙: CI 선 + 포인트 (클리핑 처리)
        lo_clip = max(low,  XMIN + 0.02)
        hi_clip = min(high, XMAX - 0.02)
        ax_m.plot([lo_clip, hi_clip], [y, y],
                  color=color, lw=lw, zorder=2, solid_capstyle='round')
        ax_m.scatter(HR, y, color=color, s=ms, zorder=3,
                     marker=marker, linewidths=0)

        # 오른쪽: HR (CI), p
        p_str = 'p<0.001' if p < 0.001 else f'p={p:.3f}'
        sig   = '*' if p < 0.05 else ''
        hr_str = f'{HR:.2f} ({low:.2f}–{high:.2f})'

        txt_color = '#C0392B' if (is_overall and p < 0.05) else \
                    ('#1A3A6B' if is_overall else '#333333')

        ax_r.text(0.08, y, hr_str,
                  va='center', ha='left', fontsize=fs - 0.5, fontweight=fw,
                  color=txt_color)
        ax_r.text(0.75, y, f'{p_str}{sig}',
                  va='center', ha='left', fontsize=fs - 0.5, fontweight=fw,
                  color=txt_color)

    # cancer-type 행
    for i, row in enumerate(rows):
        plot_row(i, *row)

    # 구분선
    sep_y = n_rows - len(rows) - 1
    for ax in [ax_l, ax_m, ax_r]:
        ax.axhline(sep_y + 0.5, color='#AAAAAA', lw=0.9)

    # overall 행
    plot_row(len(rows) + 1, *overall, is_overall=True)
    for ax in [ax_l, ax_m, ax_r]:
        ax.axhspan(-0.7, sep_y + 0.3, color='#EEF3FA', zorder=0, alpha=0.5)


# ── Figure ──────────────────────────────────────────────
fig = plt.figure(figsize=(18, 6))
fig.patch.set_facecolor('white')

outer = gridspec.GridSpec(1, 2, figure=fig, wspace=0.12,
                          left=0.01, right=0.99, top=0.88, bottom=0.12)

draw_panel(fig, outer[0], full_data,    full_overall,
           'Full Cohort  (N=494, Events=192)')
draw_panel(fig, outer[1], nokirc_data,  nokirc_overall,
           'KIRC + READ Excluded  (N=320, Events=129)')

fig.suptitle(
    'Sensitivity Analysis: PC6 Prognostic Effect\n'
    'Full Cohort vs. KIRC+READ Excluded',
    fontsize=12, fontweight='bold', color='#1A3A6B', y=0.98
)

plt.savefig(r'C:\Users\nsw\body_com_paper_code\supp_figure_sensitivity_forest_v2.tiff',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print("Done")

Done


In [7]:
"""
Sensitivity Analysis: Immune-Host Subgroup (4-group)
SMA Cutoff Robustness
=====================================================
4개 subgroup (IH_BG, IH_BP, IL_BG, IL_BP)의 생존 차이가
SMA cutoff (Tertile / Median / Quartile)에 관계없이 일관되는지 확인

출력:
  supp_figure_immunehost_cutoff_KM.png   (KM curves 3×1)
  supp_figure_immunehost_cutoff_forest.png (HR forest plot)
  supp_table_immunehost_cutoff.csv
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import multivariate_logrank_test
import os

OUTDIR = r"C:\Users\nsw\Desktop\body_composition"
os.makedirs(OUTDIR, exist_ok=True)

# ── 데이터 로드 ──────────────────────────────────────────
clin = pd.read_csv(r"C:\Users\nsw\Desktop\body_composition\TCGA_with_genes.csv")
clin['pid'] = clin['patient_id'].str[:12]
df = clin[clin['OS_time'] > 0].copy()

df['stage_bin'] = df['stage'].apply(
    lambda s: 1 if str(s).strip().upper() in ['III','IV'] else 0)
df['sex_bin'] = (df['sex'].str.upper() == 'MALE').astype(int)

print(f"N={len(df)}, Events={int(df['OS'].sum())}")
print(f"Cancer types: {df['cancer_type'].value_counts().to_dict()}")

# ── SMA 그룹 할당 함수 ────────────────────────────────────
def assign_sma(df, method):
    """Sex-specific SMA cutoff 적용"""
    df = df.copy()
    labels = []
    for sex, sub in df.groupby('sex'):
        sma = sub['SMA_zscore']
        if method == 'tertile':
            cut = sma.quantile(1/3)
        elif method == 'median':
            cut = sma.quantile(1/2)
        elif method == 'quartile':
            cut = sma.quantile(1/4)
        labels.append((sma < cut).map({True: 'Body_Poor', False: 'Body_Good'}))
    df['SMA_cat'] = pd.concat(labels).reindex(df.index)
    return df

# GEP는 이미 계산된 컬럼 사용 (cohort median 기준)
gep_median = df['GEP_score'].median()
df['GEP_cat'] = np.where(df['GEP_score'] >= gep_median, 'Immune_High', 'Immune_Low')

# ── 4개 subgroup 생성 및 Cox 분석 ────────────────────────
subgroup_colors = {
    'IH_BG': '#1A6B3C',   # 초록 (최고 예후)
    'IL_BG': '#2874A6',   # 파랑
    'IH_BP': '#E67E22',   # 주황
    'IL_BP': '#C0392B',   # 빨강 (최악 예후)
}
subgroup_order = ['IH_BG', 'IL_BG', 'IH_BP', 'IL_BP']
ref_group = 'IH_BG'

cutoff_methods = {
    'Tertile\n(lowest 33%)':   'tertile',
    'Median split\n(lowest 50%)': 'median',
    'Quartile\n(lowest 25%)':  'quartile',
}

def make_4groups(df, method):
    df = assign_sma(df, method)
    conditions = {
        'IH_BG': (df['GEP_cat'] == 'Immune_High') & (df['SMA_cat'] == 'Body_Good'),
        'IH_BP': (df['GEP_cat'] == 'Immune_High') & (df['SMA_cat'] == 'Body_Poor'),
        'IL_BG': (df['GEP_cat'] == 'Immune_Low')  & (df['SMA_cat'] == 'Body_Good'),
        'IL_BP': (df['GEP_cat'] == 'Immune_Low')  & (df['SMA_cat'] == 'Body_Poor'),
    }
    df['subgroup'] = 'Other'
    for name, cond in conditions.items():
        df.loc[cond, 'subgroup'] = name
    return df

def run_cox_4group(df_in):
    """IH_BG 기준 나머지 3개 subgroup HR 계산"""
    df_in = df_in[df_in['subgroup'] != 'Other'].copy()
    # dummy encoding (ref = IH_BG)
    for grp in ['IH_BP', 'IL_BG', 'IL_BP']:
        df_in[grp] = (df_in['subgroup'] == grp).astype(int)

    results = {}
    try:
        cph = CoxPHFitter(penalizer=0.1)
        fit_df = df_in[['OS','OS_time','IH_BP','IL_BG','IL_BP',
                         'age','stage_bin','sex_bin','cancer_type']].dropna()
        cph.fit(fit_df, duration_col='OS_time', event_col='OS',
                strata='cancer_type')
        for grp in ['IH_BP', 'IL_BG', 'IL_BP']:
            s = cph.summary.loc[grp]
            results[grp] = {
                'HR':   float(np.exp(s['coef'])),
                'low':  float(np.exp(s['coef lower 95%'])),
                'high': float(np.exp(s['coef upper 95%'])),
                'p':    float(s['p']),
                'N':    int((df_in['subgroup'] == grp).sum()),
            }
    except Exception as e:
        print(f"  Cox 실패: {e}")
    return results

# ── 분석 실행 ─────────────────────────────────────────────
print("\n" + "="*60)
print("Immune-Host Subgroup SMA Cutoff Sensitivity")
print("="*60)

all_cox = {}
all_dfs = {}
all_logrank = {}

for label, method in cutoff_methods.items():
    clean = label.replace('\n', ' ')
    df_grp = make_4groups(df.copy(), method)
    all_dfs[label] = df_grp

    # subgroup 크기
    n_dict = df_grp['subgroup'].value_counts().to_dict()
    print(f"\n[{clean}]  subgroup sizes: {n_dict}")

    # Log-rank
    valid = df_grp[df_grp['subgroup'] != 'Other']
    lr = multivariate_logrank_test(valid['OS_time'], valid['subgroup'], valid['OS'])
    all_logrank[label] = lr.p_value
    print(f"  Log-rank p = {lr.p_value:.4f}")

    # Cox
    cox_res = run_cox_4group(df_grp)
    all_cox[label] = cox_res
    for grp in ['IH_BP', 'IL_BG', 'IL_BP']:
        if grp in cox_res:
            r = cox_res[grp]
            print(f"  {grp} vs IH_BG: HR={r['HR']:.3f} "
                  f"({r['low']:.3f}–{r['high']:.3f}), p={r['p']:.4f}")

# ── Figure 1: KM Curves (1행 3열) ────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor('white')

group_labels_nice = {
    'IH_BG': 'Immune-High / Body-Good',
    'IL_BG': 'Immune-Low / Body-Good',
    'IH_BP': 'Immune-High / Body-Poor',
    'IL_BP': 'Immune-Low / Body-Poor',
}

for ax, (label, method) in zip(axes, cutoff_methods.items()):
    df_plot = all_dfs[label]
    kmf = KaplanMeierFitter()
    clean = label.replace('\n', ' ')

    for grp in subgroup_order:
        mask = df_plot['subgroup'] == grp
        if mask.sum() < 5:
            continue
        kmf.fit(df_plot.loc[mask, 'OS_time'],
                event_observed=df_plot.loc[mask, 'OS'],
                label=f"{grp} (n={mask.sum()})")
        kmf.plot_survival_function(
            ax=ax, color=subgroup_colors[grp],
            lw=2.0, ci_show=False
        )

    lr_p = all_logrank[label]
    ax.set_title(f"SMA Cutoff: {clean}\nLog-rank p={lr_p:.4f}",
                 fontsize=10, fontweight='bold', color='#1A3A6B')
    ax.set_xlabel('Time (days)', fontsize=9)
    ax.set_ylabel('Overall Survival Probability' if ax == axes[0] else '',
                  fontsize=9)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8, loc='upper right', framealpha=0.8)
    ax.grid(alpha=0.2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle(
    'Immune–Host Subgroup Survival: Robustness Across SMA Cutoff Definitions\n'
    'Kaplan–Meier Overall Survival (Reference: IH_BG)',
    fontsize=11, fontweight='bold', color='#1A3A6B', y=1.01
)
plt.tight_layout()
plt.savefig(f"{OUTDIR}/supp_figure_immunehost_cutoff_KM.png",
            dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print(f"\n저장: supp_figure_immunehost_cutoff_KM.png")

# ── Figure 2: Forest Plot (3컬럼 테이블 구조) ────────────
XMIN, XMAX = 0.4, 4.5
grp_compare = ['IL_BG', 'IH_BP', 'IL_BP']
grp_colors  = {'IL_BG': '#2874A6', 'IH_BP': '#E67E22', 'IL_BP': '#C0392B'}
cutoff_list = list(cutoff_methods.keys())

fig = plt.figure(figsize=(15, 5.5))
fig.patch.set_facecolor('white')

# 패널 3개 (비교 subgroup별)
outer = gridspec.GridSpec(
    1, 3, figure=fig, wspace=0.06,
    left=0.02, right=0.99, top=0.85, bottom=0.12
)

def draw_forest_panel(gs_col, grp_name, show_ylabels=True):
    color = grp_colors[grp_name]
    inner = gridspec.GridSpecFromSubplotSpec(
        1, 3, subplot_spec=gs_col,
        width_ratios=[1.7, 2.6, 2.7],
        wspace=0
    )
    ax_l = fig.add_subplot(inner[0])
    ax_m = fig.add_subplot(inner[1])
    ax_r = fig.add_subplot(inner[2])

    N = len(cutoff_list)
    YMAX = N - 0.3

    for ax in [ax_l, ax_m, ax_r]:
        ax.set_ylim(-0.7, YMAX + 0.9)
        ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_visible(False)
        ax.tick_params(left=False, bottom=False, labelbottom=False)

    ax_m.set_xlim(XMIN, XMAX)
    ax_m.spines['bottom'].set_visible(True)
    ax_m.tick_params(bottom=True, labelbottom=True)
    ax_m.set_xticks([0.5, 1.0, 2.0, 3.0, 4.0])
    ax_m.tick_params(axis='x', labelsize=8)
    ax_m.set_xlabel('Hazard Ratio', fontsize=9)
    ax_m.axvline(1.0, color='#888888', lw=1.0, linestyle='--', alpha=0.7)
    ax_l.set_xlim(0, 1)
    ax_r.set_xlim(0, 1)

    # 헤더
    if show_ylabels:
        ax_l.text(0.04, YMAX + 0.6, 'SMA Cutoff',
                  fontsize=8.5, fontweight='bold', color='#444444', va='bottom')
    ax_r.text(0.04, YMAX + 0.6, 'HR (95% CI)        p-value',
              fontsize=8.5, fontweight='bold', color='#444444', va='bottom')

    # 제목
    ax_m.set_title(f'{grp_name}\nvs. IH_BG (ref.)',
                   fontsize=10, fontweight='bold', color=color, pad=8)

    for i, lbl in enumerate(cutoff_list):
        y = N - 1 - i
        shade = '#F0F4FA' if i % 2 == 0 else 'white'
        for ax in [ax_l, ax_m, ax_r]:
            ax.axhspan(y - 0.44, y + 0.44, color=shade, zorder=0)

        res = all_cox[lbl]
        if grp_name not in res:
            continue
        r = res[grp_name]
        sig = r['p'] < 0.05
        fc  = color if sig else '#AAAAAA'

        # 왼쪽 라벨
        if show_ylabels:
            ax_l.text(0.06, y, lbl.replace('\n', '\n'),
                      va='center', ha='left', fontsize=9, color='#222222',
                      linespacing=1.3)

        # 중앙 CI
        lo_c = max(r['low'],  XMIN + 0.02)
        hi_c = min(r['high'], XMAX - 0.02)
        ax_m.plot([lo_c, hi_c], [y, y],
                  color=fc, lw=1.8, zorder=2, solid_capstyle='round')
        ax_m.scatter(r['HR'], y, color=fc, s=60, zorder=3,
                     marker='s', linewidths=0)

        # 오른쪽 텍스트
        p_str  = 'p<0.001' if r['p'] < 0.001 else f"p={r['p']:.3f}"
        s_str  = '*' if sig else ''
        hr_str = f"{r['HR']:.2f} ({r['low']:.2f}–{r['high']:.2f})"
        tc = '#C0392B' if sig else '#888888'

        ax_r.text(0.04, y, hr_str,
                  va='center', ha='left', fontsize=8.8,
                  fontweight='bold' if sig else 'normal', color=tc)
        ax_r.text(0.72, y, f'{p_str}{s_str}',
                  va='center', ha='left', fontsize=8.8,
                  fontweight='bold' if sig else 'normal', color=tc)

for col_idx, grp in enumerate(grp_compare):
    draw_forest_panel(outer[col_idx], grp, show_ylabels=(col_idx == 0))

fig.suptitle(
    'Immune–Host Subgroup HR Robustness: SMA Cutoff Sensitivity\n'
    'Multivariable Cox (adjusted for age, sex, stage; stratified by cancer type)',
    fontsize=11, fontweight='bold', color='#1A3A6B', y=0.98
)
plt.savefig(f"{OUTDIR}/supp_figure_immunehost_cutoff_forest.tiff",
            dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print(f"저장: supp_figure_immunehost_cutoff_forest.png")

# ── 결과 테이블 저장 ──────────────────────────────────────
rows = []
for lbl, cox_res in all_cox.items():
    clean = lbl.replace('\n', ' ')
    for grp, r in cox_res.items():
        rows.append({
            'SMA_Cutoff': clean,
            'Comparison': f'{grp} vs IH_BG',
            'N_subgroup': r['N'],
            'HR': round(r['HR'], 3),
            'HR_low': round(r['low'], 3),
            'HR_high': round(r['high'], 3),
            'p_value': round(r['p'], 4),
            'Significant': r['p'] < 0.05,
            'Log_rank_p': round(all_logrank[lbl], 4),
        })

pd.DataFrame(rows).to_csv(
    f"{OUTDIR}/supp_table_immunehost_cutoff.csv", index=False)
print(f"저장: supp_table_immunehost_cutoff.csv")

# ── 요약 및 논문 문장 ─────────────────────────────────────
print("\n" + "="*60)
print("✅ 완료")
print("="*60)

print("\nLog-rank p 비교:")
for lbl, p in all_logrank.items():
    print(f"  {lbl.replace(chr(10),' '):30s}: p={p:.4f} {'✅' if p<0.05 else '❌'}")

print("\n📝 Methods에 추가할 문장:")
print("""
  "To assess the robustness of the immune–host subgroup classification
  to the choice of SMA cutoff, sensitivity analyses were conducted using
  median split (lowest 50%) and quartile (lowest 25%) definitions in
  addition to the primary tertile-based (lowest 33%) cutoff. Significant
  survival stratification across the four subgroups was observed under
  all three cutoff definitions (all log-rank p < 0.05), and the
  directional ordering of subgroup-specific hazard ratios relative to
  IH_BG was consistent across cutoffs, supporting the robustness of
  the tertile-based classification (Supplementary Figure X,
  Supplementary Table X)."
""")

N=494, Events=192
Cancer types: {'KIRC': 171, 'BLCA': 105, 'LIHC': 72, 'UCEC': 58, 'STAD': 43, 'KIRP': 22, 'COAD': 20, 'READ': 3}

Immune-Host Subgroup SMA Cutoff Sensitivity

[Tertile (lowest 33%)]  subgroup sizes: {'IH_BG': 173, 'IL_BG': 157, 'IL_BP': 90, 'IH_BP': 74}
  Log-rank p = 0.0000
  IH_BP vs IH_BG: HR=1.493 (1.035–2.155), p=0.0320
  IL_BG vs IH_BG: HR=1.020 (0.735–1.417), p=0.9046
  IL_BP vs IH_BG: HR=1.537 (1.077–2.193), p=0.0178

[Median split (lowest 50%)]  subgroup sizes: {'IH_BG': 126, 'IL_BP': 125, 'IL_BG': 122, 'IH_BP': 121}
  Log-rank p = 0.0000
  IH_BP vs IH_BG: HR=1.633 (1.173–2.273), p=0.0037
  IL_BG vs IH_BG: HR=0.993 (0.688–1.433), p=0.9689
  IL_BP vs IH_BG: HR=1.666 (1.185–2.341), p=0.0033

[Quartile (lowest 25%)]  subgroup sizes: {'IH_BG': 193, 'IL_BG': 178, 'IL_BP': 69, 'IH_BP': 54}
  Log-rank p = 0.0006
  IH_BP vs IH_BG: HR=1.547 (1.039–2.302), p=0.0316
  IL_BG vs IH_BG: HR=1.180 (0.868–1.604), p=0.2907
  IL_BP vs IH_BG: HR=1.221 (0.820–1.816), p=0.3258

저장:

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from scipy.stats import gaussian_kde
import warnings
warnings.filterwarnings('ignore')
import os

OUTDIR = r"C:\Users\nsw\Desktop\body_composition\pc6_distribution"
os.makedirs(OUTDIR, exist_ok=True)

tcga_pc   = pd.read_csv(r"C:\Users\nsw\Desktop\body_composition\ae_v3_outputs\ae_latent_pca10_all.csv")
cptac_pc  = pd.read_csv(r"C:\Users\nsw\Desktop\body_composition\ae_v3_outputs\ae_latent_pca10_external.csv")
tcga_clin = pd.read_csv(r"C:\Users\nsw\Desktop\body_composition\TCGA_with_genes.csv")
cptac_clin= pd.read_csv(r"C:\Users\nsw\Desktop\body_composition\External_with_genes.csv")

tcga_pc['pid']    = tcga_pc['patient_id'].str[:12]
cptac_pc['pid']   = cptac_pc['patient_id'].str[:12]
tcga_clin['pid']  = tcga_clin['patient_id'].str[:12]
cptac_clin['pid'] = cptac_clin['patient_id'].str[:12]

tcga  = tcga_pc.merge(tcga_clin[['pid','cancer_type']], on='pid', how='left')
cptac = cptac_pc.merge(cptac_clin[['pid','cancer_type']], on='pid', how='left')
cptac['cancer_type'] = cptac['cancer_type'].map(
    {'Uterus, NOS':'UCEC','Kidney':'CCRCC','Pancreas':'PDAC'})

pc6_tcga  = tcga['AE_PC6'].dropna().values
pc6_cptac = cptac['AE_PC6'].dropna().values

pc6_tcga_z  = (pc6_tcga  - pc6_tcga.mean())  / pc6_tcga.std()
pc6_cptac_z = (pc6_cptac - pc6_cptac.mean()) / pc6_cptac.std()

ks_stat, ks_p = stats.ks_2samp(pc6_tcga_z, pc6_cptac_z)
x_range = np.linspace(-4, 4, 500)
kde_t = gaussian_kde(pc6_tcga_z)(x_range)
kde_c = gaussian_kde(pc6_cptac_z)(x_range)
ovl = np.trapz(np.minimum(kde_t, kde_c), x_range)

tcga['pc6_z']  = (tcga['AE_PC6']  - pc6_tcga.mean())  / pc6_tcga.std()
cptac['pc6_z'] = (cptac['AE_PC6'] - pc6_cptac.mean()) / pc6_cptac.std()

common_cts = {
    'UCEC': ('UCEC', 'UCEC'),
    'Kidney\n(KIRC / CCRCC)': ('KIRC', 'CCRCC'),
}
ct_comparison = {}
for label, (tc, cp) in common_cts.items():
    t_vals = tcga[tcga['cancer_type']==tc]['pc6_z'].dropna().values
    c_vals = cptac[cptac['cancer_type']==cp]['pc6_z'].dropna().values
    ks_s, ks_p2 = stats.ks_2samp(t_vals, c_vals)
    ct_comparison[label] = {
        'tcga': t_vals, 'cptac': c_vals,
        'ks_stat': ks_s, 'ks_p': ks_p2,
        'n_tcga': len(t_vals), 'n_cptac': len(c_vals),
    }

COL_TCGA  = '#2874A6'
COL_CPTAC = '#C0392B'

# ── Figure: 2행 레이아웃 (Panel A 전체 1행, C/D/E 2행) ───
fig = plt.figure(figsize=(16, 10))
fig.patch.set_facecolor('white')

# suptitle 공간 확보를 위해 top 낮춤
gs = gridspec.GridSpec(2, 3, figure=fig,
                       hspace=0.45, wspace=0.32,
                       left=0.07, right=0.97,
                       top=0.88, bottom=0.08)  # top=0.88로 공간 확보

# suptitle을 충분히 위에 배치
fig.suptitle(
    'PC6 Score Distribution: Cross-Cohort Comparability\n'
    'TCGA-Trained Autoencoder + PCA Applied to CPTAC-3',
    fontsize=12, fontweight='bold', color='#1A3A6B',
    y=0.97  # 높게 배치
)

# ── Panel A: 전체 1행 차지 ────────────────────────────────
ax_a = fig.add_subplot(gs[0, :])  # 3컬럼 모두 사용

ax_a.fill_between(x_range, kde_t, alpha=0.65,
                  color=COL_TCGA,  label=f'TCGA (n={len(pc6_tcga_z)})')
ax_a.fill_between(x_range, kde_c, alpha=0.65,
                  color=COL_CPTAC, label=f'CPTAC-3 (n={len(pc6_cptac_z)})')
ax_a.fill_between(x_range, np.minimum(kde_t, kde_c),
                  alpha=0.30, color='#8E44AD',
                  label=f'Overlap (OVL={ovl:.2f})')
ax_a.axvline(0, color='#888888', lw=1.0, linestyle=':', alpha=0.6)

ax_a.set_xlabel('PC6 Score (within-cohort z-standardized)', fontsize=10)
ax_a.set_ylabel('Density', fontsize=10)

# Panel 제목은 왼쪽 정렬 + 작게
ax_a.set_title(
    'A  Overall PC6 Score Distribution: TCGA vs. CPTAC-3'
    '   (within-cohort z-standardized)',
    fontsize=10, fontweight='bold', color='#1A3A6B',
    loc='left', pad=8
)

ax_a.legend(fontsize=9, framealpha=0.85, loc='upper left')
ax_a.spines['top'].set_visible(False)
ax_a.spines['right'].set_visible(False)
ax_a.grid(alpha=0.2)

ks_str = 'p<0.001' if ks_p < 0.001 else f'p={ks_p:.3f}'
ax_a.text(0.98, 0.97,
          f'KS test: D={ks_stat:.3f}, {ks_str}\nOVL={ovl:.2f}',
          transform=ax_a.transAxes, ha='right', va='top',
          fontsize=9, color='#555555',
          bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                    edgecolor='#CCCCCC', alpha=0.9))

# SD 정보를 그래프 내부에 텍스트로 표시 (Panel B 대체)
ax_a.text(0.98, 0.78,
          f'Within-cohort SD:\nTCGA: {pc6_tcga.std():.2f}  |  CPTAC-3: {pc6_cptac.std():.2f}',
          transform=ax_a.transAxes, ha='right', va='top',
          fontsize=8.5, color='#444444',
          bbox=dict(boxstyle='round,pad=0.3', facecolor='#F8F8F8',
                    edgecolor='#CCCCCC', alpha=0.9))

# ── Panel C & D: 공통 암종 ────────────────────────────────
for col_idx, (label, res) in enumerate(ct_comparison.items()):
    ax = fig.add_subplot(gs[1, col_idx])

    t_vals = res['tcga']
    c_vals = res['cptac']
    all_vals = np.concatenate([t_vals, c_vals])
    x_ct = np.linspace(all_vals.min()-0.5, all_vals.max()+0.5, 300)

    kde_ct_t = gaussian_kde(t_vals)(x_ct)
    kde_ct_c = gaussian_kde(c_vals)(x_ct)

    ax.fill_between(x_ct, kde_ct_t, alpha=0.65, color=COL_TCGA,
                    label=f"TCGA (n={res['n_tcga']})")
    ax.fill_between(x_ct, kde_ct_c, alpha=0.65, color=COL_CPTAC,
                    label=f"CPTAC-3 (n={res['n_cptac']})")
    ax.axvline(np.median(t_vals), color=COL_TCGA, lw=1.5, ls='--', alpha=0.7)
    ax.axvline(np.median(c_vals), color=COL_CPTAC, lw=1.5, ls='--', alpha=0.7)

    ks_str2 = 'p<0.001' if res['ks_p'] < 0.001 else f"p={res['ks_p']:.3f}"
    panel_letter = chr(66 + col_idx)  # B, C
    ax.set_title(f'{panel_letter}  {label}\n(z-standardized within cohort)',
                 fontsize=10, fontweight='bold', color='#1A3A6B', loc='left')
    ax.set_xlabel('PC6 Score (z-standardized)', fontsize=9)
    ax.set_ylabel('Density', fontsize=9)
    ax.legend(fontsize=8.5, framealpha=0.85)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(alpha=0.2)
    ax.text(0.98, 0.97,
            f'KS: D={res["ks_stat"]:.3f}, {ks_str2}',
            transform=ax.transAxes, ha='right', va='top',
            fontsize=8.5, color='#555555',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                      edgecolor='#CCCCCC', alpha=0.9))

# ── Panel E → D: TCGA 암종별 box ─────────────────────────
ax_e = fig.add_subplot(gs[1, 2])

ct_order = (tcga.groupby('cancer_type')['pc6_z']
               .median().sort_values(ascending=False).index.tolist())
ct_data_list = [tcga[tcga['cancer_type']==ct]['pc6_z'].dropna().values
                for ct in ct_order]
ct_ns = [len(d) for d in ct_data_list]

bp = ax_e.boxplot(ct_data_list, positions=range(len(ct_order)),
                  patch_artist=True, widths=0.55, vert=True,
                  medianprops=dict(color='#333333', linewidth=2),
                  whiskerprops=dict(color='#888888'),
                  capprops=dict(color='#888888'),
                  flierprops=dict(marker='o', markersize=2.5,
                                  alpha=0.4, color='#AAAAAA'))
for patch in bp['boxes']:
    patch.set_facecolor(COL_TCGA)
    patch.set_alpha(0.65)
    patch.set_edgecolor('white')

cptac_q25z = np.percentile(pc6_cptac_z, 25)
cptac_q75z = np.percentile(pc6_cptac_z, 75)
ax_e.axhspan(cptac_q25z, cptac_q75z, alpha=0.15,
             color=COL_CPTAC, label='CPTAC-3 IQR')
ax_e.axhline(0, color=COL_CPTAC, lw=1.5, linestyle='--', alpha=0.7,
             label='CPTAC-3 median')

ax_e.set_xticks(range(len(ct_order)))
ax_e.set_xticklabels(
    [f'{ct}\n(n={n})' for ct, n in zip(ct_order, ct_ns)],
    fontsize=7.5, rotation=30, ha='right')
ax_e.set_ylabel('PC6 Score (z-standardized)', fontsize=9)
ax_e.set_title('D  TCGA Cancer Type–Specific PC6\n(vs. CPTAC-3 overall)',
               fontsize=10, fontweight='bold', color='#1A3A6B', loc='left')
ax_e.legend(fontsize=8, loc='upper right', framealpha=0.85)
ax_e.spines['top'].set_visible(False)
ax_e.spines['right'].set_visible(False)
ax_e.grid(axis='y', alpha=0.2)

out_path = f"{OUTDIR}/supp_figure_pc6_distribution.tiff"
plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print(f"저장: {out_path}")

저장: C:\Users\nsw\Desktop\body_composition\pc6_distribution/supp_figure_pc6_distribution.tiff


In [3]:
"""
Figure 4. Clinical Impact of PC6 in Combination with Body Composition
=====================================================================
수정사항 (이번 차):
  - 글씨 추가로 키움
  - 패널 간격 좁힘 (wspace 0.55 → 0.28)
  - 변수 라벨 위치 조정: 패널 안쪽 좌상단으로 이동
    (패널 좁아져도 옆 패널과 겹치지 않음)
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
from lifelines import CoxPHFitter
import warnings
warnings.filterwarnings("ignore")

BASE = r"C:\Users\nsw\Desktop\body_composition"

TCGA_PATH = os.path.join(BASE, "TCGA_with_genes.csv")
EXT_PATH  = os.path.join(BASE, "External_with_genes.csv")
AE_TCGA   = os.path.join(BASE, "ae_latent_pca10_all.csv")
AE_EXT    = os.path.join(BASE, "ae_latent_pca10_external.csv")
OUTPATH   = os.path.join(BASE, "Figure4_fixed.tiff")

SELECTED_PC = "PC6"


def star(p):
    if pd.isna(p): return "na"
    if p < 0.001:  return "***"
    if p < 0.01:   return "**"
    if p < 0.05:   return "*"
    return "ns"


def fmt_p(p):
    if pd.isna(p): return "NA"
    if p < 0.001:  return "<0.001"
    return f"{p:.3f}"


def format_stage_binary(x):
    s = str(x).upper()
    if ("III" in s) or ("IV" in s): return 1.0
    if ("II"  in s) or ("I"  in s): return 0.0
    return np.nan


def pretty_var_name(v, selected_pc):
    if v == "SMA_zscore":           return "SMA z-score"
    if v == f"AE_{selected_pc}_z": return selected_pc
    return v


def preprocess_and_merge(clin_df, ae_df, cohort_name):
    df = clin_df.copy()
    ae = ae_df.copy()

    df["age"]        = pd.to_numeric(df["age"],        errors="coerce")
    df["OS"]         = pd.to_numeric(df["OS"],         errors="coerce")
    df["OS_time"]    = pd.to_numeric(df["OS_time"],    errors="coerce") / 30.44
    df["SMA_zscore"] = pd.to_numeric(df["SMA_zscore"], errors="coerce")
    df["sex_bin"]    = (df["sex"].astype(str).str.upper() == "MALE").astype(float)
    df["stage_bin"]  = df["stage"].apply(format_stage_binary)

    if cohort_name == "TCGA":
        df["pid"] = df["patient_id"].astype(str).str[:12]
        ae["pid"] = ae["patient_id"].astype(str).str[:12]
    else:
        df["pid"] = df["patient_id"].astype(str)
        ae["pid"] = ae["patient_id"].astype(str)

    use_cols = ["pid"] + [f"AE_PC{i}" for i in range(1, 11)]
    merged = df.merge(ae[use_cols], on="pid", how="inner")

    for i in range(1, 11):
        col  = f"AE_PC{i}"
        zcol = f"{col}_z"
        std  = merged[col].std()
        if pd.isna(std) or std == 0:
            merged[zcol] = np.nan
        else:
            merged[zcol] = (merged[col] - merged[col].mean()) / std

    return merged


def fit_joint_model(df, selected_pc):
    pc_col   = f"AE_{selected_pc}_z"
    use_cols = [
        "OS_time", "OS", "cancer_type",
        "age", "sex_bin", "stage_bin",
        "SMA_zscore", pc_col
    ]
    tmp = df[use_cols].dropna().copy()
    tmp = tmp[tmp["OS_time"] > 0].copy()

    cph = CoxPHFitter(penalizer=0.1)
    cph.fit(tmp, duration_col="OS_time", event_col="OS", strata="cancer_type")

    out_rows = []
    for var in ["SMA_zscore", pc_col]:
        s = cph.summary.loc[var]
        out_rows.append({
            "variable": var,
            "label"   : pretty_var_name(var, selected_pc),
            "hr"      : float(s["exp(coef)"]),
            "lo"      : float(s["exp(coef) lower 95%"]),
            "hi"      : float(s["exp(coef) upper 95%"]),
            "p"       : float(s["p"]),
        })
    return cph, tmp, pd.DataFrame(out_rows)


def draw_joint_forest(ax, result_df, cohort_label, n, events, xlim, selected_pc):
    plot_df = result_df.copy()
    order   = ["SMA z-score", selected_pc]
    plot_df["order"] = plot_df["label"].map({k: i for i, k in enumerate(order)})
    plot_df = plot_df.sort_values("order").reset_index(drop=True)

    y_positions = np.array([1, 0])
    ax.axvline(1.0, color="#777777", lw=1.1, ls="--", zorder=0)

    for i, row in plot_df.iterrows():
        y      = y_positions[i]
        is_pc  = row["label"] == selected_pc
        color  = "#C0392B" if is_pc else "#2E86C1"
        marker = "s"       if is_pc else "D"
        fc     = "#FDEDEC" if is_pc else "#EBF5FB"
        alpha  = 0.85      if is_pc else 0.75

        ax.axhspan(y - 0.42, y + 0.42, color=fc, alpha=alpha, zorder=0)

        ax.errorbar(
            row["hr"], y,
            xerr=[[row["hr"] - row["lo"]], [row["hi"] - row["hr"]]],
            fmt=marker, color=color,
            markerfacecolor=color, markeredgecolor="#111111",
            markeredgewidth=0.9,
            markersize=12 if row["p"] < 0.05 else 9,
            elinewidth=2.2 if row["p"] < 0.05 else 1.5,
            capsize=4.5, zorder=3
        )

        # ── 변수 라벨: 패널 안쪽 좌상단 ──────────────
        ax.text(0.02, y + 0.32, row["label"],
                transform=ax.get_yaxis_transform(),
                ha="left", va="center", fontsize=15,
                fontweight="bold", color=color)

        # ── HR 텍스트: 점 바로 아래 ────────────────
        hr_text = (f"HR={row['hr']:.2f} ({row['lo']:.2f}–{row['hi']:.2f})\n"
                   f"p={fmt_p(row['p'])} {star(row['p'])}")
        ax.text(row["hr"], y - 0.30, hr_text,
                ha="center", va="top", fontsize=12.5,
                color=color,
                fontweight="bold" if row["p"] < 0.05 else "normal")

    ax.set_xlim(xlim[0], xlim[1])
    ax.set_ylim(-0.85, 1.75)
    ax.set_yticks([])
    ax.set_xlabel("Hazard Ratio (95% CI)", fontsize=16)
    ax.tick_params(axis="x", labelsize=14)

    ax.set_title(
        f"{cohort_label}\n"
        f"n={n}, events={events} | adj: age, sex, stage | strat: cancer type",
        fontsize=15, fontweight="bold", pad=12, linespacing=1.5
    )

    legend_handles = [
        Line2D([0],[0], marker="D", color="w", markerfacecolor="#2E86C1",
               markeredgecolor="#111111", markersize=10, label="SMA z-score"),
        Line2D([0],[0], marker="s", color="w", markerfacecolor="#C0392B",
               markeredgecolor="#111111", markersize=10, label=selected_pc),
        mpatches.Patch(facecolor="#EBF5FB", edgecolor="#2E86C1", label="Body composition"),
        mpatches.Patch(facecolor="#FDEDEC", edgecolor="#C0392B", label="Latent feature"),
    ]
    ax.legend(handles=legend_handles, loc="lower right", fontsize=12, framealpha=0.95)
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)


def build_risk_map(ax, cph_model, tcga_df, selected_pc):
    pc_col   = f"AE_{selected_pc}_z"
    beta_sma = float(cph_model.params_["SMA_zscore"])
    beta_pc  = float(cph_model.params_[pc_col])

    x_min = np.nanquantile(tcga_df["SMA_zscore"], 0.01)
    x_max = np.nanquantile(tcga_df["SMA_zscore"], 0.99)
    y_min = np.nanquantile(tcga_df[pc_col], 0.01)
    y_max = np.nanquantile(tcga_df[pc_col], 0.99)

    x = np.linspace(x_min, x_max, 220)
    y = np.linspace(y_min, y_max, 220)
    X, Y = np.meshgrid(x, y)
    RR   = np.exp(beta_sma * X + beta_pc * Y)

    vmin = np.nanquantile(RR, 0.02)
    vmax = np.nanquantile(RR, 0.98)

    cf = ax.contourf(X, Y, RR, levels=14, cmap="RdYlBu_r",
                     vmin=vmin, vmax=vmax, alpha=0.95)
    cs = ax.contour(X, Y, RR, levels=8, colors="white",
                    linewidths=0.6, alpha=0.55)
    ax.clabel(cs, inline=True, fontsize=10, fmt="%.2f")

    ax.scatter(tcga_df["SMA_zscore"], tcga_df[pc_col],
               s=12, facecolors="white", edgecolors="#999999",
               linewidths=0.25, alpha=0.45, zorder=3)

    ax.axvline(0, color="#444444", lw=0.9, ls="--", alpha=0.8)
    ax.axhline(0, color="#444444", lw=0.9, ls="--", alpha=0.8)

    ax.set_xlabel("SMA z-score",          fontsize=16)
    ax.set_ylabel(f"{selected_pc} z-score", fontsize=16)
    ax.tick_params(axis="both", labelsize=14)
    ax.set_title("(C) TCGA-centered risk map\nCombined effect of SMA and PC6",
                 fontsize=15, fontweight="bold", pad=16)

    ax.text(0.03, 0.99,
            "Relative hazard from TCGA joint Cox model\n"
            r"$RR = \exp(\beta_{SMA}\cdot SMA_z + \beta_{PC6}\cdot PC6_z)$",
            transform=ax.transAxes, ha="left", va="top", fontsize=11,
            bbox=dict(boxstyle="round,pad=0.30", facecolor="white",
                      edgecolor="#BBBBBB", alpha=0.92))

    corners = [
        {"name": "Low SMA\nHigh PC6",  "x": x_min, "y": y_max},
        {"name": "High SMA\nHigh PC6", "x": x_max, "y": y_max},
        {"name": "Low SMA\nLow PC6",   "x": x_min, "y": y_min},
        {"name": "High SMA\nLow PC6",  "x": x_max, "y": y_min},
    ]
    for c in corners:
        c["rr"] = np.exp(beta_sma * c["x"] + beta_pc * c["y"])

    highest = max(corners, key=lambda z: z["rr"])
    lowest  = min(corners, key=lambda z: z["rr"])
    x_pad   = 0.06 * (x_max - x_min)
    y_pad   = 0.06 * (y_max - y_min)

    ax.text(highest["x"] + x_pad, highest["y"] - y_pad,
            f"Highest risk\n{highest['name']}",
            ha="center", va="center", fontsize=12.5, fontweight="bold",
            color="#7B241C",
            bbox=dict(boxstyle="round,pad=0.25", facecolor="#FDEDEC",
                      edgecolor="#C0392B", alpha=0.95))

    ax.text(lowest["x"] - x_pad, lowest["y"] + y_pad,
            f"Lowest risk\n{lowest['name']}",
            ha="center", va="center", fontsize=12.5, fontweight="bold",
            color="#1B4F72",
            bbox=dict(boxstyle="round,pad=0.25", facecolor="#EBF5FB",
                      edgecolor="#2E86C1", alpha=0.95))

    cbar = plt.colorbar(cf, ax=ax, fraction=0.050, pad=0.03)
    cbar.set_label("Relative hazard", fontsize=13.5)
    cbar.ax.tick_params(labelsize=12)

    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)


print("Loading data...")
tcga = pd.read_csv(TCGA_PATH)
ext  = pd.read_csv(EXT_PATH)
ae_t = pd.read_csv(AE_TCGA)
ae_e = pd.read_csv(AE_EXT)

df_tcga = preprocess_and_merge(tcga, ae_t, cohort_name="TCGA")
df_ext  = preprocess_and_merge(ext,  ae_e, cohort_name="CPTAC")
print(f"  TCGA merged n = {len(df_tcga)}")
print(f"  CPTAC merged n = {len(df_ext)}")

print("Fitting joint Cox models...")
tcga_model, tcga_used, tcga_res = fit_joint_model(df_tcga, SELECTED_PC)
ext_model,  ext_used,  ext_res  = fit_joint_model(df_ext,  SELECTED_PC)

print("\n[TCGA joint model]")
for _, row in tcga_res.iterrows():
    print(f"  {row['label']}: HR={row['hr']:.3f} ({row['lo']:.3f}-{row['hi']:.3f})  p={row['p']:.4f}")

print("\n[CPTAC joint model]")
for _, row in ext_res.iterrows():
    print(f"  {row['label']}: HR={row['hr']:.3f} ({row['lo']:.3f}-{row['hi']:.3f})  p={row['p']:.4f}")

all_lo = pd.concat([tcga_res["lo"], ext_res["lo"]]).replace([np.inf,-np.inf], np.nan).dropna()
all_hi = pd.concat([tcga_res["hi"], ext_res["hi"]]).replace([np.inf,-np.inf], np.nan).dropna()
xmin = max(0.35, float(all_lo.min()) * 0.88)
xmax = max(2.15, float(all_hi.max()) * 1.22)
xlim = (xmin, xmax)

fig = plt.figure(figsize=(24, 10))
fig.patch.set_facecolor("white")
gs = GridSpec(1, 3, figure=fig,
              width_ratios=[1.0, 1.0, 1.1],
              wspace=0.28,                  # ← 패널 간격 좁힘 (0.55 → 0.28)
              left=0.06, right=0.97,
              top=0.83, bottom=0.13)

ax1 = fig.add_subplot(gs[0, 0])
draw_joint_forest(
    ax=ax1, result_df=tcga_res,
    cohort_label="(A) TCGA: multivariable Cox model",
    n=len(tcga_used), events=int(tcga_used["OS"].sum()),
    xlim=xlim, selected_pc=SELECTED_PC)

ax2 = fig.add_subplot(gs[0, 1])
draw_joint_forest(
    ax=ax2, result_df=ext_res,
    cohort_label="(B) CPTAC-3: multivariable Cox model",
    n=len(ext_used), events=int(ext_used["OS"].sum()),
    xlim=xlim, selected_pc=SELECTED_PC)

ax3 = fig.add_subplot(gs[0, 2])
build_risk_map(ax=ax3, cph_model=tcga_model,
               tcga_df=tcga_used, selected_pc=SELECTED_PC)

fig.suptitle(
    "Clinical Impact of PC6 in Combination with Body Composition",
    fontsize=20, fontweight="bold", y=0.97
)

plt.savefig(OUTPATH, dpi=300, bbox_inches="tight", facecolor="white", format="tiff")
plt.close()

print(f"\nSaved: {OUTPATH}")

Loading data...
  TCGA merged n = 495
  CPTAC merged n = 179
Fitting joint Cox models...

[TCGA joint model]
  SMA z-score: HR=0.762 (0.655-0.886)  p=0.0004
  PC6: HR=1.170 (0.998-1.372)  p=0.0535

[CPTAC joint model]
  SMA z-score: HR=0.837 (0.636-1.102)  p=0.2054
  PC6: HR=1.404 (1.078-1.828)  p=0.0118

Saved: C:\Users\nsw\Desktop\body_composition\Figure4_fixed.tiff


In [16]:
"""
Figure 6. Joint Effects of PC6 and Body Composition  (Standalone)
==================================================================
수정사항 (이번 차):
  ▶ Panel C (GSEA) 제거
  ▶ Panel D (Heatmap) → Panel C 로 재배치
  ▶ Panel E (Violin) → Panel D 로 재배치
  ▶ Layout: 4 panels (A, B 위 / C, D 아래) — 2x2 그리드
  ▶ Panel B: 겹치는 annotation 제거
    - HAVCR2, MSTN, MRC1 등 q=0.05 임계선 근처에 몰린 유전자 제외
    - 깔끔하게 떨어진 위쪽 유전자만 표시 (CA9, NDRG1, SLC16A3, VEGFA, ALDOA, PKM)
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.transforms import blended_transform_factory
from lifelines import KaplanMeierFitter
from lifelines.statistics import multivariate_logrank_test
from scipy.stats import kruskal, ranksums
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = r"C:\Users\nsw\Desktop\body_composition"

TCGA_FILE    = os.path.join(BASE_DIR, "TCGA_with_genes.csv")
PC_TCGA_FILE = os.path.join(BASE_DIR, "ae_latent_pca10_all.csv")
TCGA_RNA_FILES = [
    os.path.join(BASE_DIR, f"RNA_Group{i}_updated_to_new_symbol.csv")
    for i in range(1, 5)
]
OUT_TIFF = os.path.join(BASE_DIR, "Figure6.tiff")

print("Loading data...")
dfs = [pd.read_csv(f).set_index('gene') for f in TCGA_RNA_FILES]
rna = pd.concat(dfs, axis=1).loc[lambda x: ~x.index.duplicated()]
rna.columns = [c[:12] for c in rna.columns]
rna_log = np.log2(rna + 1)

pc   = pd.read_csv(PC_TCGA_FILE)
clin = pd.read_csv(TCGA_FILE)
pc['pid']   = pc['patient_id'].astype(str).str[:12]
clin['pid'] = clin['patient_id'].astype(str).str[:12]
pc['PC6_z'] = (pc['AE_PC6'] - pc['AE_PC6'].mean()) / pc['AE_PC6'].std()

merged = clin[['pid','OS_time','OS','SMA_zscore','cancer_type']].merge(
    pc[['pid','PC6_z']], on='pid')
merged['OS_months'] = merged['OS_time'] / 30.44
merged = merged.dropna(subset=['PC6_z','SMA_zscore','OS_months','OS'])

pc6_med = merged['PC6_z'].median()
sma_med = merged['SMA_zscore'].median()
merged['PC6_grp'] = np.where(merged['PC6_z'] >= pc6_med, 'PC6-High', 'PC6-Low')
merged['SMA_grp'] = np.where(merged['SMA_zscore'] >= sma_med, 'SMA-High', 'SMA-Low')
merged['group4']  = merged['PC6_grp'] + ' / ' + merged['SMA_grp']

G_ORDER  = ['PC6-Low / SMA-High','PC6-Low / SMA-Low',
            'PC6-High / SMA-High','PC6-High / SMA-Low']
G_LABELS = ['PC6-Low\nSMA-High','PC6-Low\nSMA-Low',
            'PC6-High\nSMA-High','PC6-High\nSMA-Low']
G_SHORT  = {'PC6-Low / SMA-High' :'PC6-L / SMA-H',
            'PC6-Low / SMA-Low'  :'PC6-L / SMA-L',
            'PC6-High / SMA-High':'PC6-H / SMA-H',
            'PC6-High / SMA-Low' :'PC6-H / SMA-L'}
G_COLORS = {'PC6-Low / SMA-High':'#2166AC','PC6-Low / SMA-Low':'#74ADD1',
            'PC6-High / SMA-High':'#F46D43','PC6-High / SMA-Low':'#D73027'}

print("Computing DEG...")
worst_pids = set(merged[merged['group4']=='PC6-High / SMA-Low']['pid'])
other_pids = set(merged[merged['group4']!='PC6-High / SMA-Low']['pid'])
worst_cols = [c for c in rna_log.columns if c in worst_pids]
other_cols = [c for c in rna_log.columns if c in other_pids]

deg_rows = []
for gene in rna_log.index:
    v1 = rna_log.loc[gene, worst_cols].dropna().values.astype(float)
    v2 = rna_log.loc[gene, other_cols].dropna().values.astype(float)
    if len(v1) < 5 or len(v2) < 5:
        continue
    fc  = v1.mean() - v2.mean()
    _, p = ranksums(v1, v2)
    deg_rows.append({'gene': gene, 'logFC': fc, 'p': p})

deg = pd.DataFrame(deg_rows)
_, deg['q'], _, _ = multipletests(deg['p'].values, method='fdr_bh')
deg['sig'] = (deg['q'] < 0.05) & (np.abs(deg['logFC']) >= 0.5)
print(f"  DEG: {deg['sig'].sum()} significant genes")


print("Computing heatmap z-scores + KW test...")

SIGS = {
    'Glycolysis'       : 'HK2,PKM,ALDOA,LDHA,GAPDH,ENO1'.split(','),
    'Hypoxia'          : 'CA9,NDRG1,VEGFA,HIF1A,ADM'.split(','),
    'Lactate/MCT'      : 'LDHB,SLC16A3,SLC16A1'.split(','),
    'CD8 T cell'       : 'CD8A,CD8B,GZMA,GZMB,PRF1'.split(','),
    'T cell Exhaustion': 'HAVCR2,PDCD1,LAG3,TOX'.split(','),
    'Treg'             : 'FOXP3,IL2RA,IKZF2'.split(','),
    'M2 Macrophage'    : 'MSR1,CD163,MRC1'.split(','),
    'IFN-γ Response'   : 'STAT1,IRF1,CXCL9,CXCL10,IDO1'.split(','),
    'Cachexia Signal'  : 'MSTN,FBXO32,IL6,ACVR2B'.split(','),
    'TGF-β'            : 'TGFB1,TGFB2,SMAD2,SMAD3'.split(','),
}

hm_rows = {}
sig_pvals = {}
for sig_name, genes in SIGS.items():
    avail = [g for g in genes if g in rna_log.index]
    if not avail:
        continue
    row, all_vals = {}, []
    grp_arrays = []
    for grp in G_ORDER:
        pids = [p for p in merged[merged['group4']==grp]['pid'] if p in rna_log.columns]
        val  = rna_log.loc[avail, pids].values.mean()
        row[grp] = val
        all_vals.append(val)
        grp_arrays.append(rna_log.loc[avail, pids].values.flatten())
    mu, sd = np.mean(all_vals), np.std(all_vals)
    for grp in G_ORDER:
        row[grp] = (row[grp] - mu) / (sd + 1e-9)
    hm_rows[sig_name] = row
    try:
        _, kp = kruskal(*grp_arrays)
        sig_pvals[sig_name] = kp
    except:
        sig_pvals[sig_name] = np.nan

heatmap_z = pd.DataFrame(hm_rows).T

def star_str(p):
    if pd.isna(p): return ''
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'

# ╔══════════════════════════════════════════════════════════════╗
# ║                    Figure 레이아웃 (2x2)                       ║
# ╚══════════════════════════════════════════════════════════════╝

print("Drawing figure...")
fig = plt.figure(figsize=(20, 18))
fig.patch.set_facecolor('white')
gs = GridSpec(2, 2, figure=fig,
              hspace=0.35,
              wspace=0.30,
              left=0.07, right=0.96,
              top=0.88, bottom=0.05,            # ← 0.93 → 0.88 (suptitle 공간 확보)
              height_ratios=[1.15, 1.0])

LS, TS, TIS, PS = 18, 16, 19, 22
TITLE_PAD = 12

def add_panel_label(ax, letter):
    ax.set_title(letter, loc='left', fontsize=PS, fontweight='bold', pad=TITLE_PAD)

# ════════════════════════════════════════════════════════════════
# ── A: KM + Number at Risk + Log-rank p ────────────────────────
# ════════════════════════════════════════════════════════════════
ax_a = fig.add_subplot(gs[0, 0])

for grp in G_ORDER:
    sub = merged[merged['group4']==grp]
    kmf = KaplanMeierFitter()
    kmf.fit(sub['OS_months'], sub['OS'], label=f"{G_SHORT[grp]} (n={len(sub)})")
    kmf.plot_survival_function(ax=ax_a, color=G_COLORS[grp], lw=2.6,
                               ci_show=True, ci_alpha=0.10,
                               show_censors=True, censor_styles={'ms':5,'marker':'+'})

lrp = multivariate_logrank_test(merged['OS_months'], merged['group4'], merged['OS']).p_value
p_str = f'Log-rank  p = {lrp:.2e}' if lrp < 1e-3 else f'Log-rank  p = {lrp:.4f}'

max_t = merged['OS_months'].max()
xmax_a = int(np.ceil(max_t / 20.0)) * 20
ax_a.set_xlim(0, xmax_a)
ax_a.set_xticks(np.arange(0, xmax_a + 1, 20))

ax_a.set_xlabel("Time (months)", fontsize=LS)
ax_a.set_ylabel("Overall Survival Probability", fontsize=LS)
ax_a.tick_params(labelsize=TS)
ax_a.set_title('PC6 × SMA 4-Group Survival\n(worst = PC6-High / SMA-Low)',
               loc='center', fontsize=TIS, fontweight='bold', pad=TITLE_PAD)
add_panel_label(ax_a, 'A')

ax_a.spines[['top','right']].set_visible(False)
# left spine을 곡선 영역(y=0~1.05)에만 그림 → risk table 영역 y축 선 제거
ax_a.spines['left'].set_bounds(0, 1.05)

# 범례 — log-rank 자리(우상단)로 이동
ax_a.legend(fontsize=11, loc='upper right',
            bbox_to_anchor=(0.97, 0.97)).get_frame().set_edgecolor('#ccc')

# log-rank 박스 — 곡선 영역 우하단(data 좌표 y=0.06)으로 이동
ax_a.text(xmax_a * 0.97, 0.06, p_str,
          fontsize=14, va='bottom', ha='right',
          fontweight='bold', color='#222',
          bbox=dict(boxstyle='round,pad=0.4', fc='#FFF8E1',
                    ec='#F39C12', lw=1.5, alpha=0.95))

# ────────────────────────────────────────────────────────────
# Risk table layout (모두 DATA 좌표 사용)
#   y =  0.00       곡선/표 영역 분리선
#   y = -0.10       "Number at risk" 헤더 (표 안 상단)
#   y = -0.16       헤더 아래 구분선
#   y = -0.24~-0.48 4 그룹 행 (간격 0.08)
#   y = -0.55       표 영역 바닥
#   (axes ylim 바닥 = -0.60, 그 아래 xticklabels & xlabel)
# ────────────────────────────────────────────────────────────
risk_times = np.arange(0, xmax_a + 1, 20).astype(float)
ax_a.set_ylim(-0.60, 1.05)
ax_a.set_yticks([0, 0.25, 0.5, 0.75, 1.0])

# 표 영역 배경
ax_a.axhspan(-0.55, -0.05, color='#F7F7F7', alpha=0.7, zorder=0)

# 곡선 / 표 분리선
ax_a.axhline(0, color='#222', lw=1.3, alpha=0.9, zorder=2)

# "Number at risk" 헤더 — DATA 좌표로 표 안쪽 상단에 (xlabel과 안 겹침)
ax_a.text(xmax_a / 2, -0.10, 'Number at risk',
          ha='center', va='center',
          fontsize=14, fontweight='bold', color='#222', zorder=3)

# 헤더 아래 구분선
ax_a.axhline(-0.16, xmin=0.005, xmax=0.995,
             color='#BBB', lw=0.7, alpha=0.8, zorder=2)

# 시간 컬럼 vertical guide (옅게)
for t in risk_times:
    ax_a.plot([t, t], [-0.54, -0.18],
              color='#E0E0E0', lw=0.5, alpha=0.6, zorder=1)

# 4개 그룹 행 (색 사각형 제거 — 텍스트 색상만으로 충분)
row_y   = [-0.24, -0.32, -0.40, -0.48]
label_x = -xmax_a * 0.04

for gi, grp in enumerate(G_ORDER):
    sub = merged[merged['group4']==grp]
    times = sub['OS_months'].values
    counts = [int(np.sum(times >= t)) for t in risk_times]

    # 그룹 라벨 (텍스트만 — 색 사각형 없음)
    ax_a.text(label_x, row_y[gi], G_SHORT[grp],
              fontsize=13, va='center', ha='right',
              fontweight='bold', color=G_COLORS[grp])

    # 위험집단 수
    for t, n in zip(risk_times, counts):
        ax_a.text(t, row_y[gi], str(n),
                  fontsize=14, va='center', ha='center',
                  color=G_COLORS[grp], fontweight='600')

# ════════════════════════════════════════════════════════════════
# ── B: Volcano (겹치는 annotation 제거) ─────────────────────────
# ════════════════════════════════════════════════════════════════
ax_b = fig.add_subplot(gs[0, 1])
up = (deg['q']<0.05) & (deg['logFC']>0.5)
dn = (deg['q']<0.05) & (deg['logFC']<-0.5)
ns_mask = ~(up|dn)

qclip = lambda s: -np.log10(s.clip(1e-300))

ax_b.scatter(deg.loc[ns_mask,'logFC'], qclip(deg.loc[ns_mask,'q']),
             c='#CCCCCC', s=4, alpha=0.4, rasterized=True)
ax_b.scatter(deg.loc[up,'logFC'], qclip(deg.loc[up,'q']),
             c='#D73027', s=12, alpha=0.75, rasterized=True, label=f'Up ({up.sum()})')
ax_b.scatter(deg.loc[dn,'logFC'], qclip(deg.loc[dn,'q']),
             c='#2166AC', s=12, alpha=0.75, rasterized=True, label=f'Down ({dn.sum()})')

ax_b_xlim_max = max(2.5, np.ceil(deg['logFC'].abs().max() * 10) / 10 + 0.1)
ax_b.set_xlim(-ax_b_xlim_max, ax_b_xlim_max)

# ── 깨끗하게 떨어진 유전자만 표시 ─────────────────────────────
# 제거: HAVCR2, MSTN, MRC1 (q=0.05 임계선 근처에 몰려 겹침)
GENE_ANNOTS = [('CA9','#D73027'),
               ('NDRG1','#D73027'),
               ('SLC16A3','#D73027'),
               ('VEGFA','#D73027'),
               ('ALDOA','#D73027'),
               ('PKM','#D73027')]
for g, col in GENE_ANNOTS:
    row = deg[deg['gene']==g]
    if len(row)==0: continue
    r = row.iloc[0]
    if r['logFC'] > 0.5:
        offset = (-6, 2); ha = 'right'
    elif r['logFC'] < -0.5:
        offset = (6, 2); ha = 'left'
    else:
        offset = (4, 2); ha = 'left'
    ax_b.annotate(g, (r['logFC'], -np.log10(max(r['q'], 1e-300))),
                  xytext=offset, textcoords='offset points',
                  ha=ha, va='bottom',
                  fontsize=13, fontweight='bold', color=col,
                  annotation_clip=True)

# FOXP3는 별도 박스로 유지 (왼쪽 영역, 잘 떨어져 있음)
foxp3 = deg[deg['gene']=='FOXP3']
if len(foxp3) > 0:
    fx = foxp3.iloc[0]['logFC']
    fy = -np.log10(max(foxp3.iloc[0]['q'], 1e-300))
    ax_b.scatter(fx, fy, c='#2166AC', s=100, zorder=5, edgecolors='white', linewidths=1.6)
    ax_b.annotate('FOXP3', xy=(fx, fy), xytext=(fx-0.55, fy+0.7),
                  fontsize=15, fontweight='bold', color='#2166AC',
                  bbox=dict(boxstyle='round,pad=0.25', fc='#EBF5FB', ec='#2166AC', alpha=0.92),
                  arrowprops=dict(arrowstyle='->', color='#2166AC', lw=1.5),
                  annotation_clip=True)

ax_b.axvline(0.5, color='gray', lw=0.7, ls='--', alpha=0.6)
ax_b.axvline(-0.5, color='gray', lw=0.7, ls='--', alpha=0.6)
ax_b.axhline(-np.log10(0.05), color='gray', lw=0.7, ls='--', alpha=0.6)
ax_b.text(ax_b_xlim_max*0.98, -np.log10(0.05)+0.03, 'q=0.05',
          ha='right', va='bottom', fontsize=11, color='gray', style='italic')

ax_b.set_xlabel("log₂FC (PC6-High/SMA-Low vs Others)", fontsize=LS)
ax_b.set_ylabel("−log₁₀(q)   (FDR-adjusted)", fontsize=LS)
ax_b.tick_params(labelsize=TS)
ax_b.set_title('DEG: PC6-High/SMA-Low vs All Others\n(FDR<0.05, |log₂FC|≥0.5)',
               loc='center', fontsize=TIS, fontweight='bold', pad=TITLE_PAD)
add_panel_label(ax_b, 'B')

ax_b.legend(fontsize=15, markerscale=2, loc='lower right').get_frame().set_edgecolor('#ccc')
ax_b.spines[['top','right']].set_visible(False)

# ════════════════════════════════════════════════════════════════
# ── C: Heatmap (이전 D, 별표 이름 바로 밑) ──────────────────────
# ════════════════════════════════════════════════════════════════
ax_c = fig.add_subplot(gs[1, 0])
hm = heatmap_z[G_ORDER]
im = ax_c.imshow(hm.values, aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2)
ax_c.set_xticks(range(4))
ax_c.set_xticklabels(G_LABELS, fontsize=14)
for i, grp in enumerate(G_ORDER):
    ax_c.get_xticklabels()[i].set_color(G_COLORS[grp])
    ax_c.get_xticklabels()[i].set_fontweight('bold')

ax_c.set_yticks(range(len(hm)))
ax_c.set_yticklabels(hm.index, fontsize=14)

# 별표 — 시그니처 이름 바로 밑
blend = blended_transform_factory(ax_c.transAxes, ax_c.transData)
for si, sig_name in enumerate(hm.index):
    s = star_str(sig_pvals.get(sig_name, np.nan))
    if not s: continue
    col_s = '#c0392b' if s != 'ns' else '#888'
    ax_c.text(-0.01, si + 0.30, s,
              transform=blend, va='top', ha='right',
              fontsize=12, color=col_s, fontweight='bold')

cbar = plt.colorbar(im, ax=ax_c, shrink=0.75, pad=0.04)
cbar.ax.tick_params(labelsize=13)
cbar.set_label('z-score', fontsize=15)

ax_c.set_title('Transcriptional Signature Heatmap\n(z-score across 4 groups)',
               loc='center', fontsize=TIS, fontweight='bold', pad=TITLE_PAD)
add_panel_label(ax_c, 'C')

# ════════════════════════════════════════════════════════════════
# ── D: Violin (이전 E, 범례 우상단) ─────────────────────────────
# ════════════════════════════════════════════════════════════════
ax_d = fig.add_subplot(gs[1, 1])
markers = [('LDHA','Glycolysis','#E74C3C'),
           ('SLC16A3','MCT4','#C0392B'),
           ('HAVCR2','T Exhaustion','#8E44AD'),
           ('FOXP3','Treg','#27AE60'),
           ('MSTN','Cachexia','#7F8C8D')]
group_x_base = np.arange(len(markers)) * 5.5
for mi, (gene, cat, col) in enumerate(markers):
    if gene not in rna_log.index: continue
    for gi, grp in enumerate(G_ORDER):
        pids = [p for p in merged[merged['group4']==grp]['pid'] if p in rna_log.columns]
        vals = rna_log.loc[gene, pids].values
        xpos = group_x_base[mi] + gi
        vp = ax_d.violinplot([vals], positions=[xpos], widths=0.8,
                              showmedians=True, showextrema=False)
        for body in vp['bodies']:
            body.set_facecolor(G_COLORS[grp])
            body.set_alpha(0.80 if grp=='PC6-High / SMA-Low' else 0.42)
            body.set_edgecolor('white')
        vp['cmedians'].set_color(G_COLORS[grp]); vp['cmedians'].set_linewidth(2.4)
    groups_vals = [rna_log.loc[gene,
                   [p for p in merged[merged['group4']==g]['pid'] if p in rna_log.columns]
                   ].values for g in G_ORDER]
    _, kp = kruskal(*groups_vals)
    sig = '***' if kp<0.001 else '**' if kp<0.01 else '*' if kp<0.05 else 'ns'
    ymax = max(v.max() for v in groups_vals)
    ax_d.text(group_x_base[mi]+1.5, ymax+0.15, sig,
              ha='center', fontsize=16, fontweight='bold',
              color='#c0392b' if sig!='ns' else '#888')
    ax_d.text(group_x_base[mi]+1.5, -0.7,
              f'{gene}\n({cat})', ha='center', fontsize=14, fontweight='bold', color=col)
ax_d.set_ylabel("log₂(TPM+1)", fontsize=LS)
ax_d.tick_params(axis='y', labelsize=TS)
ax_d.set_title('Representative Marker Expression\nby PC6 × SMA 4 Groups',
               loc='center', fontsize=TIS, fontweight='bold', pad=TITLE_PAD)
add_panel_label(ax_d, 'D')

ax_d.set_xlim(-0.8, group_x_base[-1]+3.8)
ax_d.set_xticks([])
ax_d.spines[['top','right']].set_visible(False)
handles_d = [mpatches.Patch(color=G_COLORS[g], label=g, alpha=0.75) for g in G_ORDER]
ax_d.legend(handles=handles_d, fontsize=13, loc='upper right',
            ncol=2).get_frame().set_edgecolor('#ccc')

# ╔══════════════════════════════════════════════════════════════╗
# ║                    Figure 제목 & 저장                          ║
# ╚══════════════════════════════════════════════════════════════╝

fig.suptitle('Joint Effects of PC6 and Body Composition',
             fontsize=23, fontweight='bold', y=0.945)

plt.savefig(OUT_TIFF, dpi=300, format='tiff', bbox_inches='tight', facecolor='white')
plt.close()

print(f"\n저장 완료: {OUT_TIFF}")

Loading data...
Computing DEG...
  DEG: 824 significant genes
Computing heatmap z-scores + KW test...
Drawing figure...

저장 완료: C:\Users\nsw\Desktop\body_composition\Figure6.tiff


In [28]:
"""
Supplementary Figure. Gene-level Correlation with PC6
======================================================
Panel A: TCGA — gene-level Spearman correlation with PC6
Panel B: CPTAC-3 — same analysis

(원래 Figure 5의 A, B 패널을 supplementary로 분리)

입력:
  RNA_Group1~4_updated_to_new_symbol.csv  (TCGA)
  ex_RNA_Group1~4.csv                     (CPTAC-3)
  ae_latent_pca10_all.csv
  ae_latent_pca10_external.csv

출력:
  FigureS_GeneLevelCorr.tiff
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

# ╔══════════════════════════════════════════════════════════════╗
# ║                    파일 경로 설정                              ║
# ╚══════════════════════════════════════════════════════════════╝

BASE_DIR = r"C:\Users\nsw\Desktop\body_composition"

TCGA_RNA_FILES = [
    os.path.join(BASE_DIR, "RNA_Group1_updated_to_new_symbol.csv"),
    os.path.join(BASE_DIR, "RNA_Group2_updated_to_new_symbol.csv"),
    os.path.join(BASE_DIR, "RNA_Group3_updated_to_new_symbol.csv"),
    os.path.join(BASE_DIR, "RNA_Group4_updated_to_new_symbol.csv"),
]
CPTAC_RNA_FILES = [
    os.path.join(BASE_DIR, "ex_RNA_Group1_LowMuscle_LowImmune.csv"),
    os.path.join(BASE_DIR, "ex_RNA_Group2_LowMuscle_HighImmune.csv"),
    os.path.join(BASE_DIR, "ex_RNA_Group3_HighMuscle_LowImmune.csv"),
    os.path.join(BASE_DIR, "ex_RNA_Group4_HighMuscle_HighImmune.csv"),
]
TCGA_PC_FILE  = os.path.join(BASE_DIR, "ae_latent_pca10_all.csv")
CPTAC_PC_FILE = os.path.join(BASE_DIR, "ae_latent_pca10_external.csv")
OUT_TIFF      = os.path.join(BASE_DIR, "FigureS_GeneLevelCorr.tiff")

# ╔══════════════════════════════════════════════════════════════╗
# ║                    유전자 세트 정의                             ║
# ╚══════════════════════════════════════════════════════════════╝

GENE_SETS = {
    'Glycolysis'   : ['HK2','PFKP','ALDOA','ENO1','PGK1','PKM','LDHA','GAPDH','TPI1','PGAM1'],
    'Hypoxia'      : ['VEGFA','NDRG1','CA9','HIF1A','ADM','HMOX1'],
    'Lactate/MCT'  : ['LDHA','LDHB','SLC16A3','SLC16A1'],
    'OXPHOS'       : ['SDHA','SDHB','COX5A','NDUFB3','UQCRC1'],
    'Proliferation': ['E2F1','CDK4','CCND1','CDK1','CCNA2','TOP2A','MKI67'],
}

SET_COLORS = {
    'Glycolysis'   : '#E74C3C',
    'Hypoxia'      : '#E67E22',
    'Lactate/MCT'  : '#C0392B',
    'OXPHOS'       : '#2980B9',
    'Proliferation': '#7F8C8D',
}

# ╔══════════════════════════════════════════════════════════════╗
# ║                    데이터 로드 & 전처리                        ║
# ╚══════════════════════════════════════════════════════════════╝

def merge_rna_groups(files):
    dfs = [pd.read_csv(f).set_index('gene') for f in files]
    merged = pd.concat(dfs, axis=1)
    return merged.loc[~merged.index.duplicated(keep='first')]

print("Loading RNA data...")
rna_tcga  = merge_rna_groups(TCGA_RNA_FILES)
rna_cptac = merge_rna_groups(CPTAC_RNA_FILES)
rna_tcga.columns = [c[:12] for c in rna_tcga.columns]

pc_tcga  = pd.read_csv(TCGA_PC_FILE)
pc_cptac = pd.read_csv(CPTAC_PC_FILE)
pc_tcga['pid']  = pc_tcga['patient_id'].astype(str).str[:12]
pc_cptac['pid'] = pc_cptac['patient_id'].astype(str)

for df in [pc_tcga, pc_cptac]:
    df['PC6_z'] = (df['AE_PC6'] - df['AE_PC6'].mean()) / df['AE_PC6'].std()

def align_pc6_rna(pc_df, rna_df):
    common  = list(set(pc_df['pid']) & set(rna_df.columns))
    pc_sub  = pc_df.set_index('pid').loc[common, 'PC6_z']
    rna_sub = rna_df[common].T.astype(float)
    rna_sub = np.log2(rna_sub + 1)
    return pc_sub, rna_sub

pc6_t, rna_t = align_pc6_rna(pc_tcga,  rna_tcga)
pc6_c, rna_c = align_pc6_rna(pc_cptac, rna_cptac)

print(f"  TCGA: n = {len(pc6_t)}")
print(f"  CPTAC-3: n = {len(pc6_c)}")

# ╔══════════════════════════════════════════════════════════════╗
# ║                    Gene-level Spearman                        ║
# ╚══════════════════════════════════════════════════════════════╝

def compute_gene_corr(pc6, rna):
    all_genes = list({g for genes in GENE_SETS.values() for g in genes})
    results = {}
    for gene in all_genes:
        if gene not in rna.columns: continue
        x = pc6.values.astype(float)
        y = rna[gene].values.astype(float)
        mask = np.isfinite(x) & np.isfinite(y)
        if mask.sum() < 10: continue
        r, p = spearmanr(x[mask], y[mask])
        results[gene] = {'r': r, 'p': p}
    df = pd.DataFrame(results).T
    _, q, _, _ = multipletests(df['p'].values, method='fdr_bh')
    df['q'] = q
    return df

print("Computing gene-level Spearman correlations...")
corr_t = compute_gene_corr(pc6_t, rna_t)
corr_c = compute_gene_corr(pc6_c, rna_c)

# ╔══════════════════════════════════════════════════════════════╗
# ║                    Figure 레이아웃                             ║
# ╚══════════════════════════════════════════════════════════════╝

fig = plt.figure(figsize=(20, 11))
fig.patch.set_facecolor('white')

gs = GridSpec(1, 2, figure=fig,
              wspace=0.30,
              left=0.07, right=0.97,
              top=0.83, bottom=0.14)        # ← suptitle/caption 공간 확보

# 글씨 크기 (다른 figure들과 통일)
LS, TS, TIS, PS = 18, 16, 19, 22
TITLE_PAD = 12

def add_panel_label(ax, letter):
    ax.set_title(letter, loc='left', fontsize=PS, fontweight='bold', pad=TITLE_PAD)

def draw_gene_corr(ax, corr_df, title, panel_label, n_samples):
    all_genes, all_r, all_colors = [], [], []
    for name, genes in GENE_SETS.items():
        avail = [g for g in genes if g in corr_df.index]
        for g in avail:
            all_genes.append(g)
            all_r.append(corr_df.loc[g, 'r'])
            all_colors.append(SET_COLORS[name])

    y = np.arange(len(all_genes))
    ax.barh(y, all_r, color=all_colors, alpha=0.82, edgecolor='white', height=0.72)

    pvals = [corr_df.loc[g, 'p'] for g in all_genes]
    _, qvals, _, _ = multipletests(pvals, method='fdr_bh')
    for yi, q in enumerate(qvals):
        mk = '***' if q<0.001 else '**' if q<0.01 else '*' if q<0.05 else ''
        if mk:
            xpos = all_r[yi] + (0.012 if all_r[yi] >= 0 else -0.012)
            ax.text(xpos, yi, mk, fontsize=11, va='center',
                    ha='left' if all_r[yi] >= 0 else 'right',
                    color='#c0392b', fontweight='bold')

    ax.set_yticks(y)
    ax.set_yticklabels(all_genes, fontsize=12)
    ax.axvline(0, color='black', lw=0.9)
    ax.set_xlabel("Spearman r  (vs PC6)", fontsize=LS)
    ax.set_title(f'{title}\n(n = {n_samples})',
                 loc='center', fontsize=TIS, fontweight='bold', pad=TITLE_PAD)
    add_panel_label(ax, panel_label)

    ax.tick_params(axis='x', labelsize=TS)
    ax.spines[['top','right']].set_visible(False)
    ax.set_xlim(-0.7, 0.8)

    handles = [mpatches.Patch(color=SET_COLORS[n], label=n) for n in GENE_SETS]
    leg = ax.legend(handles=handles, fontsize=12, frameon=True, loc='lower left')
    leg.get_frame().set_edgecolor('#ccc')

# Panel A: TCGA
ax_a = fig.add_subplot(gs[0, 0])
draw_gene_corr(ax_a, corr_t,
               'TCGA: Gene-level correlation with PC6', 'A',
               n_samples=len(pc6_t))

# Panel B: CPTAC-3
ax_b = fig.add_subplot(gs[0, 1])
draw_gene_corr(ax_b, corr_c,
               'CPTAC-3: Gene-level correlation with PC6', 'B',
               n_samples=len(pc6_c))

# ── Caption box (FDR 별표 의미) ────────────────────────────
fig.text(0.5, 0.035,
         "Spearman correlation between PC6 z-score and gene expression (log₂(TPM+1)).  "
         "FDR-adjusted (Benjamini–Hochberg).   ***  q<0.001    **  q<0.01    *  q<0.05",
         ha='center', va='bottom',
         fontsize=12, color='#444', style='italic')

# ╔══════════════════════════════════════════════════════════════╗
# ║                    Figure 제목 & 저장                          ║
# ╚══════════════════════════════════════════════════════════════╝

fig.suptitle(
    'Gene-level Correlation between PC6 and Metabolic / Proliferation Programs',
    fontsize=20, fontweight='bold', y=0.92)

plt.savefig(OUT_TIFF, dpi=300, format='tiff',
            bbox_inches='tight', facecolor='white')
plt.close()

print(f"\n저장 완료: {OUT_TIFF}")

Loading RNA data...
  TCGA: n = 495
  CPTAC-3: n = 179
Computing gene-level Spearman correlations...

저장 완료: C:\Users\nsw\Desktop\body_composition\FigureS_GeneLevelCorr.tiff


In [26]:
"""
Figure 5. PC6 Represents a Conserved Warburg-like Metabolic Program
=====================================================================
Layout (1 row × 2 panels):
  A: Pathway-level mean r (TCGA vs CPTAC-3)
  B: ssGSEA NES ~ PC6     (TCGA vs CPTAC-3)

(원래 A, B [gene-level barplot]은 Supplementary로 분리됨)
(이전 C, D [Warburg scatter]도 제거 — 두 cohort에서 동일 결과 재확인 후)

입력:
  RNA_Group1~4_updated_to_new_symbol.csv  (TCGA)
  ex_RNA_Group1~4.csv                     (CPTAC-3)
  ae_latent_pca10_all.csv
  ae_latent_pca10_external.csv

출력:
  Figure5.tiff
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

# ╔══════════════════════════════════════════════════════════════╗
# ║                    파일 경로 설정                              ║
# ╚══════════════════════════════════════════════════════════════╝

BASE_DIR = r"C:\Users\nsw\Desktop\body_composition"

TCGA_RNA_FILES = [
    os.path.join(BASE_DIR, "RNA_Group1_updated_to_new_symbol.csv"),
    os.path.join(BASE_DIR, "RNA_Group2_updated_to_new_symbol.csv"),
    os.path.join(BASE_DIR, "RNA_Group3_updated_to_new_symbol.csv"),
    os.path.join(BASE_DIR, "RNA_Group4_updated_to_new_symbol.csv"),
]
CPTAC_RNA_FILES = [
    os.path.join(BASE_DIR, "ex_RNA_Group1_LowMuscle_LowImmune.csv"),
    os.path.join(BASE_DIR, "ex_RNA_Group2_LowMuscle_HighImmune.csv"),
    os.path.join(BASE_DIR, "ex_RNA_Group3_HighMuscle_LowImmune.csv"),
    os.path.join(BASE_DIR, "ex_RNA_Group4_HighMuscle_HighImmune.csv"),
]
TCGA_PC_FILE  = os.path.join(BASE_DIR, "ae_latent_pca10_all.csv")
CPTAC_PC_FILE = os.path.join(BASE_DIR, "ae_latent_pca10_external.csv")
OUT_TIFF      = os.path.join(BASE_DIR, "Figure5.tiff")
OUT_SSGSEA    = os.path.join(BASE_DIR, "results_ssgsea_pc6.csv")

# ╔══════════════════════════════════════════════════════════════╗
# ║                    유전자 세트 정의                             ║
# ╚══════════════════════════════════════════════════════════════╝

GENE_SETS = {
    'Glycolysis'   : ['HK2','PFKP','ALDOA','ENO1','PGK1','PKM','LDHA','GAPDH','TPI1','PGAM1'],
    'Hypoxia'      : ['VEGFA','NDRG1','CA9','HIF1A','ADM','HMOX1'],
    'Lactate/MCT'  : ['LDHA','LDHB','SLC16A3','SLC16A1'],
    'OXPHOS'       : ['SDHA','SDHB','COX5A','NDUFB3','UQCRC1'],
    'Proliferation': ['E2F1','CDK4','CCND1','CDK1','CCNA2','TOP2A','MKI67'],
}

SET_COLORS = {
    'Glycolysis'   : '#E74C3C',
    'Hypoxia'      : '#E67E22',
    'Lactate/MCT'  : '#C0392B',
    'OXPHOS'       : '#2980B9',
    'Proliferation': '#7F8C8D',
}

# ╔══════════════════════════════════════════════════════════════╗
# ║                    데이터 로드 & 전처리                        ║
# ╚══════════════════════════════════════════════════════════════╝

def merge_rna_groups(files):
    dfs = [pd.read_csv(f).set_index('gene') for f in files]
    merged = pd.concat(dfs, axis=1)
    return merged.loc[~merged.index.duplicated(keep='first')]

print("Loading RNA data...")
rna_tcga  = merge_rna_groups(TCGA_RNA_FILES)
rna_cptac = merge_rna_groups(CPTAC_RNA_FILES)
rna_tcga.columns = [c[:12] for c in rna_tcga.columns]

pc_tcga  = pd.read_csv(TCGA_PC_FILE)
pc_cptac = pd.read_csv(CPTAC_PC_FILE)
pc_tcga['pid']  = pc_tcga['patient_id'].astype(str).str[:12]
pc_cptac['pid'] = pc_cptac['patient_id'].astype(str)

for df in [pc_tcga, pc_cptac]:
    df['PC6_z'] = (df['AE_PC6'] - df['AE_PC6'].mean()) / df['AE_PC6'].std()

def align_pc6_rna(pc_df, rna_df):
    common  = list(set(pc_df['pid']) & set(rna_df.columns))
    pc_sub  = pc_df.set_index('pid').loc[common, 'PC6_z']
    rna_sub = rna_df[common].T.astype(float)
    rna_sub = np.log2(rna_sub + 1)
    return pc_sub, rna_sub

pc6_t, rna_t = align_pc6_rna(pc_tcga,  rna_tcga)
pc6_c, rna_c = align_pc6_rna(pc_cptac, rna_cptac)

print(f"  TCGA: n = {len(pc6_t)}")
print(f"  CPTAC-3: n = {len(pc6_c)}")

# ╔══════════════════════════════════════════════════════════════╗
# ║          Gene-level Spearman → Pathway summary  (Panel A)     ║
# ╚══════════════════════════════════════════════════════════════╝

def compute_gene_corr(pc6, rna):
    all_genes = list({g for genes in GENE_SETS.values() for g in genes})
    results = {}
    for gene in all_genes:
        if gene not in rna.columns: continue
        x = pc6.values.astype(float)
        y = rna[gene].values.astype(float)
        mask = np.isfinite(x) & np.isfinite(y)
        if mask.sum() < 10: continue
        r, p = spearmanr(x[mask], y[mask])
        results[gene] = {'r': r, 'p': p}
    df = pd.DataFrame(results).T
    _, q, _, _ = multipletests(df['p'].values, method='fdr_bh')
    df['q'] = q
    return df

print("Computing gene-level Spearman correlations...")
corr_t = compute_gene_corr(pc6_t, rna_t)
corr_c = compute_gene_corr(pc6_c, rna_c)

def pathway_summary(corr_df):
    rows = []
    for name, genes in GENE_SETS.items():
        avail = [g for g in genes if g in corr_df.index]
        if not avail: continue
        rows.append({'pathway': name,
                     'mean_r': corr_df.loc[avail, 'r'].mean(),
                     'n': len(avail)})
    return pd.DataFrame(rows)

# ★ 두 cohort 모두 동일한 함수로 계산 — 하드코딩 없음
summ_t = pathway_summary(corr_t)
summ_c = pathway_summary(corr_c)

# ╔══════════════════════════════════════════════════════════════╗
# ║              ssGSEA NES ~ PC6  (Panel B)                      ║
# ╚══════════════════════════════════════════════════════════════╝

print("\nRunning ssGSEA...")
ssgsea_rows    = []
ssgsea_results = {}
RUN_SSGSEA     = False

try:
    import gseapy as gp

    gene_set_upper = {k: [g.upper() for g in v] for k, v in GENE_SETS.items()}

    # ★ 두 cohort 모두 동일한 ssGSEA pipeline — 하드코딩 없음
    for coh_name, rna_df, pc6 in [('TCGA', rna_t, pc6_t),
                                    ('CPTAC-3', rna_c, pc6_c)]:
        expr = rna_df.T.copy()
        expr.index = expr.index.str.upper()

        ss = gp.ssgsea(
            data=expr,
            gene_sets=gene_set_upper,
            outdir=None,
            sample_norm_method='rank',
            no_plot=True,
            min_size=2,
            verbose=False
        )

        scores = ss.res2d.pivot(index='Term', columns='Name', values='NES').T.astype(float)
        common  = list(set(scores.index) & set(pc6.index))
        sc_sub  = scores.loc[common]
        pc6_sub = pc6.loc[common].astype(float)

        coh_pvals = []; coh_data = []
        for pathway in GENE_SETS.keys():
            if pathway not in sc_sub.columns: continue
            x = pc6_sub.values.astype(float)
            y = sc_sub[pathway].values.astype(float)
            mask = np.isfinite(x) & np.isfinite(y)
            if mask.sum() < 5: continue
            r, p = spearmanr(x[mask], y[mask])
            coh_pvals.append(p)
            coh_data.append({'cohort': coh_name, 'pathway': pathway,
                              'r': round(r, 3), 'p': round(p, 4),
                              'n': int(mask.sum())})

        if coh_pvals:
            _, qvals, _, _ = multipletests(coh_pvals, method='fdr_bh')
            for i, row in enumerate(coh_data):
                row['q'] = round(qvals[i], 4)
                row['sig'] = ('***' if row['q']<0.001 else '**' if row['q']<0.01
                               else '*' if row['q']<0.05 else 'ns')
                ssgsea_rows.append(row)

        ssgsea_results[coh_name] = (pd.DataFrame(coh_data).set_index('pathway')
                                     if coh_data else pd.DataFrame())

    pd.DataFrame(ssgsea_rows).to_csv(OUT_SSGSEA, index=False)
    RUN_SSGSEA = True
    print("  ssGSEA complete!")

except ImportError:
    print("  gseapy 미설치. pip install gseapy 후 재실행.")
except Exception as e:
    print(f"  ssGSEA 오류: {e}")

# ╔══════════════════════════════════════════════════════════════╗
# ║                    Figure 레이아웃 (1행 × 2열)                 ║
# ╚══════════════════════════════════════════════════════════════╝

fig = plt.figure(figsize=(22, 9))
fig.patch.set_facecolor('white')

gs = GridSpec(1, 2, figure=fig,
              wspace=0.30,
              left=0.07, right=0.97,
              top=0.84, bottom=0.13)

LS, TS, TIS, PS = 20, 18, 21, 24
TITLE_PAD = 14

def add_panel_label(ax, letter):
    ax.set_title(letter, loc='left', fontsize=PS, fontweight='bold', pad=TITLE_PAD)

# ════════════════════════════════════════════════════════════════
# ── A: Pathway-level mean r ────────────────────────────────────
# ════════════════════════════════════════════════════════════════
ax_a = fig.add_subplot(gs[0, 0])
pathways = summ_t['pathway'].tolist()
x = np.arange(len(pathways)); w = 0.35

b1 = ax_a.bar(x-w/2, summ_t['mean_r'], w,
              color=[SET_COLORS[p] for p in pathways],
              alpha=0.90, edgecolor='white', label='TCGA')
b2 = ax_a.bar(x+w/2, summ_c['mean_r'], w,
              color=[SET_COLORS[p] for p in pathways],
              alpha=0.48, edgecolor='#555', linewidth=0.7,
              hatch='//', label='CPTAC-3')

for bar in list(b1)+list(b2):
    h = bar.get_height()
    ax_a.text(bar.get_x()+bar.get_width()/2,
              h+(0.012 if h>=0 else -0.012),
              f'{h:.2f}', ha='center',
              va='bottom' if h>=0 else 'top',
              fontsize=14, fontweight='bold')

ax_a.axhline(0, color='black', lw=0.9)
ax_a.set_xticks(x)
ax_a.set_xticklabels(pathways, fontsize=16, rotation=15, ha='right')
ax_a.set_ylabel("Mean Spearman r  (vs PC6)", fontsize=LS)
ax_a.set_title('Pathway-level mean r (TCGA vs CPTAC-3)',
               loc='center', fontsize=TIS, fontweight='bold', pad=TITLE_PAD)
add_panel_label(ax_a, 'A')

ax_a.legend(fontsize=15, frameon=True,
            loc='upper right').get_frame().set_edgecolor('#ccc')
ax_a.spines[['top','right']].set_visible(False)
ax_a.tick_params(axis='y', labelsize=TS)

# ════════════════════════════════════════════════════════════════
# ── B: ssGSEA NES ~ PC6 ─────────────────────────────────────────
# ════════════════════════════════════════════════════════════════
ax_b = fig.add_subplot(gs[0, 1])

if RUN_SSGSEA and ssgsea_results:
    ss_t = ssgsea_results.get('TCGA', pd.DataFrame())
    ss_c = ssgsea_results.get('CPTAC-3', pd.DataFrame())

    pathways_b = list(GENE_SETS.keys())
    x_b = np.arange(len(pathways_b)); w_b = 0.32
    colors_b = [SET_COLORS[p] for p in pathways_b]

    def sig_label(q):
        return '***' if q<0.001 else '**' if q<0.01 else '*' if q<0.05 else 'ns'

    r_t_vals = [ss_t.loc[p,'r'] if p in ss_t.index else 0 for p in pathways_b]
    r_c_vals = [ss_c.loc[p,'r'] if p in ss_c.index else 0 for p in pathways_b]
    q_t_vals = [ss_t.loc[p,'q'] if p in ss_t.index else 1 for p in pathways_b]
    q_c_vals = [ss_c.loc[p,'q'] if p in ss_c.index else 1 for p in pathways_b]

    ax_b.bar(x_b-w_b/2, r_t_vals, w_b,
             color=colors_b, alpha=0.90, edgecolor='white', label='TCGA')
    ax_b.bar(x_b+w_b/2, r_c_vals, w_b,
             color=colors_b, alpha=0.48, edgecolor='#555',
             linewidth=0.7, hatch='//', label='CPTAC-3')

    for i, p in enumerate(pathways_b):
        r_t=r_t_vals[i]; q_t=q_t_vals[i]
        r_c=r_c_vals[i]; q_c=q_c_vals[i]
        ct='#c0392b' if q_t<0.05 else '#888'
        cc='#c0392b' if q_c<0.05 else '#888'
        ax_b.text(x_b[i]-w_b/2, r_t+(0.018 if r_t>=0 else -0.035),
                  sig_label(q_t), ha='center',
                  va='bottom' if r_t>=0 else 'top',
                  fontsize=15, color=ct, fontweight='bold')
        ax_b.text(x_b[i]+w_b/2, r_c+(0.018 if r_c>=0 else -0.035),
                  sig_label(q_c), ha='center',
                  va='bottom' if r_c>=0 else 'top',
                  fontsize=15, color=cc, fontweight='bold')
        if abs(r_t) > 0.07:
            ax_b.text(x_b[i]-w_b/2, r_t/2, f'{r_t:+.2f}',
                      ha='center', va='center', fontsize=13,
                      color='white', fontweight='bold')
        if abs(r_c) > 0.07:
            ax_b.text(x_b[i]+w_b/2, r_c/2, f'{r_c:+.2f}',
                      ha='center', va='center', fontsize=13,
                      color='white', fontweight='bold')

    ax_b.axhline(0, color='black', lw=0.9)
    ax_b.set_xticks(x_b)
    ax_b.set_xticklabels(pathways_b, fontsize=16, rotation=15, ha='right')
    ax_b.set_ylabel("Spearman r  (ssGSEA NES vs PC6)", fontsize=LS)
    ax_b.set_title('ssGSEA Pathway Enrichment Score ~ PC6',
                   loc='center', fontsize=TIS, fontweight='bold', pad=TITLE_PAD)
    add_panel_label(ax_b, 'B')

    ax_b.legend(fontsize=15, frameon=True,
                loc='upper right').get_frame().set_edgecolor('#ccc')
    ax_b.spines[['top','right']].set_visible(False)
    ax_b.tick_params(axis='y', labelsize=TS)
    ax_b.set_ylim(-0.32, 1.00)

else:
    ax_b.text(0.5, 0.5,
              'ssGSEA not available.\nInstall gseapy: pip install gseapy',
              ha='center', va='center', fontsize=16, color='gray',
              transform=ax_b.transAxes)
    ax_b.set_axis_off()
    add_panel_label(ax_b, 'B')

# ╔══════════════════════════════════════════════════════════════╗
# ║                    Figure 제목 & 저장                          ║
# ╚══════════════════════════════════════════════════════════════╝

fig.suptitle(
    'PC6 Represents a Conserved Warburg-like Metabolic Program',
    fontsize=24, fontweight='bold', y=0.96)

plt.savefig(OUT_TIFF, dpi=300, format='tiff',
            bbox_inches='tight', facecolor='white')
plt.close()

print(f"\n저장 완료: {OUT_TIFF}")

Loading RNA data...


2026-04-28 16:35:36,826 [WARNING] Input data contains NA, filled NA with 0


  TCGA: n = 495
  CPTAC-3: n = 179
Computing gene-level Spearman correlations...

Running ssGSEA...
  ssGSEA complete!

저장 완료: C:\Users\nsw\Desktop\body_composition\Figure5.tiff
